# Section 3: Answer with sources

Goal: Dock answers customs questions only from Harbor Freight's handbook, cites the sections it used,
says so when the handbook has no answer, and we pick the smallest model that still gets it right.

## Step 1. Look at the handbook and the questions

In [1]:
import json

handbook = json.loads(open("dock/handbook.json").read())
questions = json.loads(open("dock/questions.json").read())
for doc in handbook[:4]:
    print(doc["id"], doc["title"], "-", len(doc["text"].split()), "words")
print("...", len(handbook), "sections")
print()
for q in questions[:2] + questions[-2:]:
    print(q["id"], q["question"], "->", q["sources"] or "not in the handbook")

HB-01 Documents for every import - 51 words
HB-02 Certificate of origin - 56 words
HB-03 Coffee and other food into the EU - 51 words
HB-04 Wooden furniture and packaging - 50 words
... 12 sections

Q01 Can we import roasted coffee into Germany, and what must the invoice show? -> ['HB-03']
Q02 Shipment HF-26038 is on customs hold in Hamburg. Which three documents does the broker need? -> ['HB-01']
Q13 What is the import duty rate for bicycle frames into the Netherlands? -> not in the handbook
Q14 Does Harbor Freight ship live animals? -> not in the handbook


## Step 2. Connect to Mistral

In [2]:
import os
from dotenv import load_dotenv
from mistralai.client import Mistral
from mistralai.client.utils import BackoffStrategy, RetryConfig

load_dotenv()
retry = RetryConfig("backoff", BackoffStrategy(1000, 20000, 2.0, 120000),
                    retry_connection_errors=True)
client = Mistral(api_key=os.environ["MISTRAL_API_KEY"], retry_config=retry,
                 timeout_ms=120000)
print("key loaded:", len(os.environ["MISTRAL_API_KEY"]) > 0)

key loaded: True


## Step 3. Turn text into vectors

In [3]:
import numpy as np

def embed(texts):
    r = client.embeddings.create(model="mistral-embed", inputs=texts)
    return np.array([d.embedding for d in r.data]), r.usage.total_tokens

doc_vectors, used = embed([d["title"] + ". " + d["text"] for d in handbook])
print(doc_vectors.shape, "-", used, "tokens")
print(np.round(doc_vectors[0][:5], 4), "...")
print("length of each vector:", np.round(np.linalg.norm(doc_vectors[0]), 4))

(12, 1024) - 913 tokens
[-0.051   0.0119  0.0247  0.0433  0.063 ] ...
length of each vector: 1.0


## Step 4. Find the closest sections

In [4]:
def search(question, k=3):
    q_vector, _ = embed([question])
    scores = doc_vectors @ q_vector[0]
    best = np.argsort(-scores)[:k]
    return [(handbook[i]["id"], round(float(scores[i]), 3)) for i in best]

print(questions[0]["question"])
for doc_id, score in search(questions[0]["question"]):
    print("  ", doc_id, score)

Can we import roasted coffee into Germany, and what must the invoice show?


   HB-03 0.856
   HB-04 0.744
   HB-10 0.742


In [5]:
hits1 = hits3 = 0
for q in questions:
    found = search(q["question"])
    ids = [doc_id for doc_id, _ in found]
    if q["sources"]:
        hits1 += ids[0] in q["sources"]
        hits3 += any(s in ids for s in q["sources"])
    else:
        print(q["id"], "not in handbook, best match:", found[0])
answerable = sum(1 for q in questions if q["sources"])
print(f"top-1: {hits1}/{answerable}   top-3: {hits3}/{answerable}")

Q13 not in handbook, best match: ('HB-02', 0.734)


Q14 not in handbook, best match: ('HB-05', 0.798)
top-1: 12/12   top-3: 12/12


## Step 5. Answer only from the sources

In [6]:
from pydantic import BaseModel, Field

class Answer(BaseModel):
    found: bool = Field(description="false if the sections do not answer the question")
    answer: str = Field(description="Short answer, using only the sections")
    sources: list[str] = Field(description="Ids of the sections you used, like HB-03")

RULES = ("You answer questions for Harbor Freight's ops team. Use ONLY the handbook "
         "sections below. If they don't answer the question, set found to false.")


def ask(question, model="mistral-small-latest", k=3):
    found = search(question, k)
    context = "\n\n".join(f"[{d['id']}] {d['title']}: {d['text']}"
                           for d in handbook if d["id"] in dict(found))
    r = client.chat.parse(model=model, response_format=Answer, messages=[
        {"role": "system", "content": RULES + "\n\n" + context},
        {"role": "user", "content": question}])
    return r.choices[0].message.parsed, [i for i, _ in found], r.usage.total_tokens

a, retrieved, tokens = ask(questions[1]["question"])
print(a.model_dump_json(indent=1))
print("retrieved:", retrieved, "-", tokens, "tokens")

{
 "found": true,
 "answer": "The commercial invoice, the packing list and the bill of lading.",
 "sources": [
  "HB-01"
 ]
}
retrieved: ['HB-01', 'HB-06', 'HB-11'] - 329 tokens


## Step 6. Check every answer in code

In [7]:
def check(q, a, retrieved):
    problems = []
    if not q["sources"]:
        return [] if not a.found else ["answered a question the handbook can't answer"]
    if not a.found:
        return ["said not found, but the handbook has it"]
    if not set(a.sources) <= set(retrieved):
        problems.append(f"cites {a.sources}, but only {retrieved} were given")
    if not set(q["sources"]) & set(a.sources):
        problems.append(f"should cite {q['sources']}")
    missing = [w for w in q["must_contain"]           # "four|4" = either spelling
               if not any(alt in a.answer.lower() for alt in w.split("|"))]
    if missing:
        problems.append(f"answer lacks {missing}")
    return problems


results = {}
for q in questions:
    a, retrieved, tokens = ask(q["question"])
    results[q["id"]] = check(q, a, retrieved)
    print(q["id"], "OK" if not results[q["id"]] else results[q["id"]], "|", a.answer[:70])
print("correct:", sum(not p for p in results.values()), "/", len(questions))

Q01 OK | Yes, roasted coffee may be imported into the EU. The commercial invoic


Q02 OK | The broker needs the commercial invoice, the packing list and the bill


Q03 OK | Certificate of origin


Q04 OK | ISPM 15 mark


Q05 ['said not found, but the handbook has it'] | No


...

Q10 OK | An EORI number starting with GB


Q11 OK | At least 24 hours before the cargo is loaded at the origin port.


Q12 OK | 26 tonnes


Q13 OK | 


Q14 OK | 
correct: 12 / 14


## Step 7. The smallest model that works

In [8]:
import time

for model in ["ministral-8b-latest", "mistral-small-latest", "mistral-medium-latest"]:
    correct = tokens_total = 0
    start = time.time()
    failures = []
    for q in questions:
        a, retrieved, tokens = ask(q["question"], model=model)
        problems = check(q, a, retrieved)
        correct += not problems
        tokens_total += tokens
        if problems:
            failures.append(q["id"])
    seconds = (time.time() - start) / len(questions)
    print(f"{model:22} {correct}/{len(questions)}  {tokens_total} tokens"
          f"  {seconds:.1f} s/question  failed: {failures}")

ministral-8b-latest    11/14  4447 tokens  1.9 s/question  failed: ['Q02', 'Q05', 'Q09']


mistral-small-latest   12/14  4377 tokens  1.5 s/question  failed: ['Q05', 'Q09']


mistral-medium-latest  14/14  4471 tokens  1.5 s/question  failed: []


## Step 8. Fix the schema, then compare again

In [9]:
class AnswerV2(Answer):
    found: bool = Field(description="true if a rule in the sections answers the question, "
                                    "also when you must apply it, like a limit or a deadline. "
                                    "false only if no section covers the topic")

Answer = AnswerV2
for model in ["ministral-8b-latest", "mistral-small-latest"]:
    failures = [q["id"] for q in questions
                if check(q, *ask(q["question"], model=model)[:2])]
    print(f"{model:22} {len(questions) - len(failures)}/{len(questions)}  failed: {failures}")

ministral-8b-latest    11/14  failed: ['Q02', 'Q05', 'Q09']


mistral-small-latest   13/14  failed: ['Q09']
